<a href="https://colab.research.google.com/github/AjayBora17/Weather-prediction/blob/main/Weather_PRIDICTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

section 1:Import Libraries

In [192]:
import requests # This library helps us to fetch data from API
import pandas as pd # pandas used for analyzing and handling data
import numpy as np #its for numerical operations
from sklearn.model_selection import train_test_split #to split data into training and testing sets
from sklearn.preprocessing import LabelEncoder #to convert categorical data into numerical data
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor #model for classification
from sklearn.metrics import mean_squared_error #to measur accuracy of our prediction
from datetime import datetime, timedelta #to handle date and time
import pytz

In [193]:
API_KEY = 'fcbfe7d41eafa69240e1d60f7c2946c0' #replace with your actual key
BASE_URL = 'https://api.openweathermap.org/data/2.5/' #base url for making API requests

1. Fetch Current Weather Data

In [194]:
def get_current_weather(city):
  url = f"{BASE_URL}weather?q={city}&appid={API_KEY}&units=metric" #construct the api request url
  response = requests.get(url) #send the get request to api
  data = response.json()
  return{
      'city': data['name'],
      'current_temp': round(data['main']['temp']),
      'feels_like' : round(data['main']['feels_like']),
      'temp_min': round(data['main']['temp_min']),
      'temp_max': round(data['main']['temp_max']),
      'humidity': round(data['main']['humidity']),
      'description': data['weather'][0]['description'],
      'country': data['sys']['country'],
      'Wind_Gust_Dir': data['wind']['deg'],
      'pressure': data['main']['pressure'],
      'Wind_Gust_Speed': data['wind']['speed']

  }

2. Read Historical Data

In [195]:
def read_historical_data(filename):
  df = pd.read_csv(filename) #reads csv files into dataframe
  df = df.dropna() #remove rows with duplicate values
  df = df.drop_duplicates()
  return df

3. Prepare data for training

In [196]:
def prepare_data(data):
  le = LabelEncoder() #creates a labelencoder instance
  data['WindGustDir'] = le.fit_transform(data['WindGustDir'])
  data['RainTomorrow'] = le.fit_transform(data['RainTomorrow'])

  #define the feature variable and target variable
  x = data[['MinTemp', 'MaxTemp', 'WindGustDir', 'WindGustSpeed', 'Humidity', 'Pressure', 'Temp']] #feature variables to predict target
  y = data['RainTomorrow'] #target variable whether it will rain or not

  return x,y,le #return all



4. Train Rain Prediction Model

In [197]:
def train_rain_model(x,y):
  x_train, x_test,  y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=42)
  model = RandomForestClassifier(n_estimators=100, random_state=42)
  model.fit(x_train, y_train) #train the model

  y_pred = model.predict(x_test) #to make prediction on test set

  print("Mean Squared Error for Rain  Model")

  print(mean_squared_error(y_test, y_pred))

  return model

5. Prepare regression data

In [198]:
def prepare_regression_data(data, feature):
  x,y= [],[] #initialize list for feature and target values

  for i in range(len(data)-1):
    x.append(data[feature].iloc[i])
    y.append(data[feature].iloc[i+1])

  x= np.array(x).reshape(-1,1)
  y= np.array(y)
  return x,y


6. Train Regression Model

In [199]:
def train_regression_model(x,y):
  model = RandomForestRegressor(n_estimators=100, random_state=42)
  model.fit(x,y)
  return model

7. Predict Future

In [200]:
def predict_future(model, current_value):
  predictions = [current_value]

  for i in range(5):
    next_value = model.predict(np.array([[predictions[-1]]]))

    predictions.append(next_value[0])
  return predictions[1:]

8. Weather Analysis Function

In [206]:
def weather_view():
  city = input('Enter any city name:')
  try:
    current_weather = get_current_weather(city)
  except KeyError as e:
    print(f"Error fetching weather for '{city}': {e}. Please check the city name and try again.")
    weather_view() # Recursively call to ask for city again
    return # Exit current call

  #load historical data
  historical_data = read_historical_data('/content/weather.csv')

  #prepare and train the rain prediction model

  x,y,le = prepare_data(historical_data)

  rain_model = train_rain_model(x,y)

  #map wind direction to compass points

  wind_deg = current_weather['Wind_Gust_Dir'] % 360
  compass_points = [
      ("N",0,11.25),("NNE",11.25,33.75),("NE",33.75,56.25),
      ("ENE",56.25,78.75),("E",78.75,101.25),("ESE",101.25,123.75),
      ("SE",123.75,146.25),("SSE",146.25,168.75),("S",168.75,191.25),
      ("SSW",191.25,213.75),("SW",213.75,236.25),("WSW",236.25,258.75),
      ("W",258.75,281.25),("WNW",281.25,303.75),("NW",303.75,326.25),
      ("NNW",326.25,348.75),
      ("N", 348.75, 360) # Restored to cover the remaining part of North
  ]
  compass_direction = next(point for point, start, end in compass_points if start <= wind_deg <end)

  compass_direction_encoded = le.transform([compass_direction])[0] if compass_direction in le.classes_ else -1

  current_data={
      'MinTemp': current_weather['temp_min'],
      'MaxTemp': current_weather['temp_max'],
      'WindGustDir': compass_direction_encoded,
      'WindGustSpeed':current_weather['Wind_Gust_Speed'],
      'Humidity':current_weather['humidity'],
      'Pressure':current_weather['pressure'],
      'Temp':current_weather['current_temp']
  }

  current_df = pd.DataFrame([current_data])

  #rain prediction

  rain_prediction = rain_model.predict( current_df)[0]

  #prepare regression model for temperature abd humidity

  x_temp, y_temp = prepare_regression_data(historical_data, 'Temp')

  x_hum, y_hum = prepare_regression_data(historical_data, 'Humidity')

  temp_model = train_regression_model(x_temp, y_temp)

  hum_model = train_regression_model(x_hum, y_hum)

  #predict future temp and humidity

  future_temp = predict_future(temp_model, current_weather['temp_min'])
  future_humidity = predict_future(hum_model, current_weather['humidity'])

  #prepare time for future prediction

  timezone = pytz.timezone('Asia/Kolkata')
  now = datetime.now(timezone)
  next_hour = now + timedelta(hours=1)
  next_hour = next_hour.replace(minute=0, second=0, microsecond=0)

  future_times = [(next_hour + timedelta(hours=i)).strftime("%H:00") for i in range(5)]

  #display results

  print(f"City: {city},{current_weather['country']}") # Changed from current_data['country']
  print(f"Current Temperature: {current_weather['current_temp']}")
  print(f"Feels Like: {current_weather['feels_like']}")
  print(f"Minimum Temperature: {current_weather['temp_min']}°C")
  print(f"Maximum Temperature: {current_weather['temp_max']}°C")
  print(f"Humidity: {current_weather['humidity']}%")
  print(f"Weather Prediction: {current_weather['description']}")
  print(f"Rain Prediction: {'Yes' if rain_prediction else 'No'}")

  print("\nFuture Temperature Prediction:")

  for time, temp in zip(future_times, future_temp):
    print(f"{time}: {round(temp, 1)}°C")

  print("\nFuture Humidity Prediction:")

  for time, humidity in zip(future_times, future_humidity):
      print(f"{time}: {round(humidity, 1)}%")


  weather_view()


Enter any city name:kolkata
Mean Squared Error for Rain  Model
0.1506849315068493
City: kolkata,IN
Current Temperature: 28
Feels Like: 34
Minimum Temperature: 28°C
Maximum Temperature: 28°C
Humidity: 94%
Weather Prediction: overcast clouds
Rain Prediction: Yes

Future Temperature Prediction:
08:00: 27.7°C
09:00: 22.8°C
10:00: 25.4°C
11:00: 24.7°C
12:00: 23.4°C

Future Humidity Prediction:
08:00: 62.2%
09:00: 56.5%
10:00: 40.7%
11:00: 52.9%
12:00: 51.0%
Enter any city name:uttarakhand
Mean Squared Error for Rain  Model
0.1506849315068493
City: uttarakhand,IN
Current Temperature: 20
Feels Like: 20
Minimum Temperature: 20°C
Maximum Temperature: 20°C
Humidity: 93%
Weather Prediction: overcast clouds
Rain Prediction: Yes

Future Temperature Prediction:
08:00: 22.1°C
09:00: 23.3°C
10:00: 19.0°C
11:00: 24.5°C
12:00: 24.5°C

Future Humidity Prediction:
08:00: 55.8%
09:00: 45.2%
10:00: 46.3%
11:00: 48.2%
12:00: 53.7%
Enter any city name:kerala
Mean Squared Error for Rain  Model
0.15068493150684

KeyboardInterrupt: Interrupted by user